# 02 — Semantic Retrieval

Learn how to find the source passage most relevant to an LLM claim.

## 1. Why Semantic Retrieval?

We need to find the source text most relevant to each claim. Embeddings help us compare text by meaning instead of exact words.

## 2. Imports

Import the libraries needed for embeddings and similarity.

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## 3. Load the Embedding Model

Load a pretrained model that converts sentences into embeddings.

In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 4. Understanding Embeddings

Convert a sentence into a numerical vector.

In [3]:
sentence = "The telescope was launched in 2021."

embedding = model.encode(sentence)

print("Embedding:")
print(embedding)
print("\nNumber of dimensions:", len(embedding))

Embedding:
[ 1.49112474e-02 -1.21799614e-02  3.20056006e-02  3.25925201e-02
 -3.36584076e-02 -7.34718889e-02 -1.13479272e-01 -7.83277489e-03
 -9.17665288e-02  8.09789523e-02  2.18245052e-02 -1.56965833e-02
  3.78635600e-02  1.60569865e-02 -4.63841890e-04  1.84753686e-02
  1.95129886e-02 -1.41869292e-01 -2.21070973e-03 -8.68999287e-02
 -2.55322363e-02  6.16982132e-02 -4.77201603e-02  2.39611976e-02
  3.56490128e-02  1.77050177e-02 -6.69460893e-02 -8.58548656e-02
  5.56028355e-03  5.79576530e-02  2.05866657e-02  4.80680615e-02
 -1.18353873e-01 -8.74448661e-03 -3.53888310e-02  1.99772697e-02
  8.64897668e-03 -8.42196345e-02  4.87456433e-02 -6.41782507e-02
 -2.93138660e-02 -1.31596550e-01 -1.16362218e-02  2.02694889e-02
  2.73998212e-02 -3.68913612e-03 -4.60348949e-02 -3.11706699e-02
  2.36710645e-02  2.41477471e-02 -9.80037674e-02 -7.07742870e-02
  2.53320355e-02 -1.23553745e-01  2.79554203e-02  6.97036013e-02
 -3.48289870e-02 -3.37579548e-02  5.27697466e-02 -2.90475599e-02
  7.70976359e-

### Observation

The sentence is represented as a 384-dimensional vector.

## 5. Measuring Semantic Similarity

Compare two sentence embeddings using cosine similarity.

In [4]:
sentence1 = "The telescope was launched in 2021."
sentence2 = "The telescope began its mission in 2021."

embedding1 = model.encode(sentence1)
embedding2 = model.encode(sentence2)

similarity = cosine_similarity(
    [embedding1],
    [embedding2]
)[0][0]

print("Cosine similarity:", similarity)

Cosine similarity: 0.9349685


## 6. Compare with an Unrelated Sentence

To understand the similarity score better, we compare the telescope sentence with an unrelated sentence.

In [5]:
sentence3 = "The capital of France is Paris."

embedding3 = model.encode(sentence3)

unrelated_similarity = cosine_similarity(
    [embedding1],
    [embedding3]
)[0][0]

print("Similarity with unrelated sentence:", unrelated_similarity)

Similarity with unrelated sentence: 0.13984194


### Observation

Similarity helps us find relevant text. It does not tell us whether a claim is true.

## 7. Document Chunking

Split the source document into smaller chunks for retrieval.

In [6]:
chunks = [
    "The James Webb Space Telescope is a space telescope designed to conduct infrared astronomy.",

    "It was launched on December 25, 2021, aboard an Ariane 5 rocket from the Guiana Space Centre.",

    "The telescope operates near the second Lagrange point, approximately 1.5 million kilometers from Earth."
]

print("Number of chunks:", len(chunks))

Number of chunks: 3


## 8. Embed Source Chunks

Convert every source chunk into an embedding.

In [7]:
chunk_embeddings = model.encode(chunks)

print("Embedding matrix shape:", chunk_embeddings.shape)

Embedding matrix shape: (3, 384)


### Observation

Three chunks produce an embedding matrix with shape `(3, 384)`.

## 9. Retrieve Evidence for a Claim

Find the source chunk most similar to the claim.

In [8]:
claim = "The telescope was launched in 2021."

claim_embedding = model.encode(claim)

similarities = cosine_similarity(
    [claim_embedding],
    chunk_embeddings
)[0]

for i, score in enumerate(similarities):
    print(f"Chunk {i}: {score:.4f}")

Chunk 0: 0.4936
Chunk 1: 0.5136
Chunk 2: 0.5113


## 10. Select the Most Relevant Chunk

Choose the chunk with the highest similarity score.

In [9]:
best_index = similarities.argmax()

print("Best chunk index:", best_index)
print("Evidence:", chunks[best_index])
print("Similarity:", similarities[best_index])

Best chunk index: 1
Evidence: It was launched on December 25, 2021, aboard an Ariane 5 rocket from the Guiana Space Centre.
Similarity: 0.5135689


## 11. Create a Reusable Retrieval Function

Combine the retrieval steps into one function.

In [10]:
def retrieve_evidence(claim, chunks):
    claim_embedding = model.encode(claim)
    chunk_embeddings = model.encode(chunks)

    similarities = cosine_similarity(
        [claim_embedding],
        chunk_embeddings
    )[0]

    best_index = similarities.argmax()

    return {
        "claim": claim,
        "evidence": chunks[best_index],
        "similarity": similarities[best_index]
    }

## 12. Test the Retrieval Function

Test the function with one claim.

In [11]:
result = retrieve_evidence(
    "The telescope was launched in 2021.",
    chunks
)

print("Claim:", result["claim"])
print("Evidence:", result["evidence"])
print("Similarity:", result["similarity"])

Claim: The telescope was launched in 2021.
Evidence: It was launched on December 25, 2021, aboard an Ariane 5 rocket from the Guiana Space Centre.
Similarity: 0.5135689


## 13. Test Multiple Claims

Test retrieval on several claims, including an intentionally incorrect claim.

In [12]:
claims = [
    "The telescope was launched in 2021.",
    "The telescope was launched using an Ariane 5 rocket.",
    "The telescope is approximately 1.5 million kilometers from Earth.",
    "The telescope was launched from India."
]

for claim in claims:
    result = retrieve_evidence(claim, chunks)

    print("\nCLAIM:")
    print(result["claim"])

    print("\nEVIDENCE:")
    print(result["evidence"])

    print("\nSIMILARITY:")
    print(f"{result['similarity']:.4f}")

    print("-" * 60)


CLAIM:
The telescope was launched in 2021.

EVIDENCE:
It was launched on December 25, 2021, aboard an Ariane 5 rocket from the Guiana Space Centre.

SIMILARITY:
0.5136
------------------------------------------------------------

CLAIM:
The telescope was launched using an Ariane 5 rocket.

EVIDENCE:
It was launched on December 25, 2021, aboard an Ariane 5 rocket from the Guiana Space Centre.

SIMILARITY:
0.6180
------------------------------------------------------------

CLAIM:
The telescope is approximately 1.5 million kilometers from Earth.

EVIDENCE:
The telescope operates near the second Lagrange point, approximately 1.5 million kilometers from Earth.

SIMILARITY:
0.8093
------------------------------------------------------------

CLAIM:
The telescope was launched from India.

EVIDENCE:
The telescope operates near the second Lagrange point, approximately 1.5 million kilometers from Earth.

SIMILARITY:
0.5198
------------------------------------------------------------


## 14. Next Step

Next, use an NLI model to decide whether the retrieved evidence supports, contradicts, or does not establish the claim.